# Prepare Base Model — Corrected Version

This version keeps the tutor's workflow but fixes notebook execution order, working-directory handling, and layer freezing.

For your dataset, `params.yaml` should use `CLASSES: 3` for Healthy, Coccidiosis, and Salmonella.


In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path(r"F:\Chicken-Disease-Classification")

if not (PROJECT_ROOT / "config" / "config.yaml").exists():
    raise FileNotFoundError(f"config.yaml not found under: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())
print("config.yaml exists:", Path("config/config.yaml").exists())
print("params.yaml exists:", Path("params.yaml").exists())


Working directory: F:\Chicken-Disease-Classification
config.yaml exists: True
params.yaml exists: True


In [2]:
from dataclasses import dataclass
from pathlib import Path

from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories


In [3]:
@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int


In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        return PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )


In [9]:
import os
import ssl
import certifi

cert_path = certifi.where()

os.environ["SSL_CERT_FILE"] = cert_path
os.environ["REQUESTS_CA_BUNDLE"] = cert_path

ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=cert_path
)

print("Using certificate:", cert_path)

Using certificate: c:\Users\feroz\miniconda3\envs\cnncls\lib\site-packages\certifi\cacert.pem


In [11]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.12.0


In [12]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        path.parent.mkdir(parents=True, exist_ok=True)
        model.save(path)

    def get_base_model(self):
        self.model = tf.keras.applications.vgg16.VGG16(
            input_shape=tuple(self.config.params_image_size),
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(
            path=self.config.base_model_path,
            model=self.model
        )

    @staticmethod
    def _prepare_full_model(
        model,
        classes,
        freeze_all,
        freeze_till,
        learning_rate
    ):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)

        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(
                learning_rate=learning_rate
            ),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(
            path=self.config.updated_base_model_path,
            model=self.full_model
        )


In [13]:
config = ConfigurationManager()
prepare_base_model_config = config.get_prepare_base_model_config()

print("Image size:", prepare_base_model_config.params_image_size)
print("Classes:", prepare_base_model_config.params_classes)
print("Weights:", prepare_base_model_config.params_weights)
print("Learning rate:", prepare_base_model_config.params_learning_rate)

if prepare_base_model_config.params_classes != 3:
    raise ValueError(
        "For Healthy, Coccidiosis and Salmonella, params.yaml must contain CLASSES: 3"
    )


[2026-08-25 00:57:31,687: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-25 00:57:31,689: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-25 00:57:31,691: INFO: common: created directory at: artifacts]
[2026-08-25 00:57:31,692: INFO: common: created directory at: artifacts/prepare_base_model]
Image size: [224, 224, 3]
Classes: 3
Weights: imagenet
Learning rate: 0.001


In [14]:
try:
    prepare_base_model = PrepareBaseModel(
        config=prepare_base_model_config
    )

    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()

    print("Prepare base model stage completed successfully.")

except Exception as e:
    raise e


58889256/58889256 [==============================] - 8s 0us/step
[2026-08-25 00:57:44,741: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     7385

## If `SSLError: [ASN1: NOT_ENOUGH_DATA]` still appears

That error occurs while Keras is trying to download the ImageNet VGG16 weights. It is an SSL/certificate problem in the Python/Conda environment, not a problem in the model code above.

Keep `WEIGHTS: imagenet` for transfer learning. Repair the environment certificates, restart VS Code/kernel, and then rerun the notebook from the top.

Terminal commands:

```bash
conda activate cnncls
conda install --force-reinstall ca-certificates certifi openssl -y
python -m pip install --upgrade certifi
```
